# Gradient Boosting: California Housing regression

This notebook uses scikit-learn's `GradientBoostingRegressor` to predict median house value from all California Housing features.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, loss='squared_error', random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {np.sqrt(mse):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.4f}')
print(f'R²:   {r2_score(y_test, y_pred):.4f}')

importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
order = importance.importances_mean.argsort()
plt.figure(figsize=(8, 5))
plt.barh(X.columns[order], importance.importances_mean[order])
plt.xlabel('Decrease in test R² after permutation')
plt.title('Gradient Boosting: permutation feature importance')
plt.show()

## Gradient Boosting: model, loss, and evaluation

### Formula notation

- $\mathbf{x}_i$: California Housing features for district $i$; $y_i$: true median house value.
- $F_m(\mathbf{x})$: ensemble prediction after $m$ trees; $h_m(\mathbf{x})$: new regression tree at iteration $m$.
- $M$: number of trees (`n_estimators`); $\eta$: learning rate; $r_{im}$: residual for example $i$ at iteration $m$.

### Boosting model

Gradient Boosting builds trees sequentially. It starts with a constant prediction, usually the training-target mean:

$$F_0(\mathbf{x}) = \frac{1}{n}\sum_{i=1}^{n}y_i$$

For squared-error regression, the negative gradient is the residual:

$$r_{im} = y_i - F_{m-1}(\mathbf{x}_i)$$

A new tree fits these residuals, then the ensemble updates:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta h_m(\mathbf{x})$$

This procedure is used because each later tree focuses on the mistakes made by earlier trees. A smaller learning rate makes updates more cautious and usually requires more trees.

### Training objective and test evaluation

$$\mathrm{MSE} = \frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2$$

$$\mathrm{RMSE} = \sqrt{\mathrm{MSE}}, \qquad \mathrm{MAE} = \frac{1}{m}\sum_{i=1}^{m}|y_i-\hat{y}_i|$$

$$R^2 = 1 - \frac{\sum_{i=1}^{m}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{m}(y_i-\bar{y})^2}$$

**Minimize** MSE, RMSE, and MAE: $0$ is best and there is no fixed worst value. **Maximize** $R^2$: $1$ is best, $0$ equals predicting the test mean, and negative values are worse than that baseline.